In [1]:
import pandas as pd
#import pandas libraries

df = pd.read_csv("C:/Users/APTIKA/OneDrive/Documents/RIMA/customer_repeat_order/data/raw/online_retail_II.csv", parse_dates = ["InvoiceDate"])
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
print(df.shape, df.dtypes, sep="/")

(1067371, 8)/Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object


In [3]:
df["Customer ID"] = df["Customer ID"].astype("Int64")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [4]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [5]:
print("duplikat :", df.duplicated().sum())

duplikat : 34335


In [6]:
print(df["InvoiceDate"].min(), df["InvoiceDate"].max())

2009-12-01 07:45:00 2011-12-09 12:50:00


In [7]:
df[["Quantity", "Price"]].describe(percentiles=[.01, .5, .99])

,Quantity,Price
count,1.067371e+06,1.067371e+06
mean,9.938898e+00,4.649388e+00
std,1.727058e+02,1.235531e+02
min,-8.099500e+04,-5.359436e+04
1%,-3.000000e+00,2.100000e-01
50%,3.000000e+00,2.100000e+00
99%,1.000000e+02,1.800000e+01
max,8.099500e+04,3.897000e+04


In [8]:
print("Jumlah Cancel : ", df["Invoice"].str.startswith("C").sum())

Jumlah Cancel :  19494


In [9]:
print("Jumlah Quantity < 0 :", (df["Quantity"]<=0).sum())
print("Price < 0 :", (df["Price"]<=0).sum())

Jumlah Quantity < 0 : 22950
Price < 0 : 6207


In [10]:
df.loc[~df["StockCode"].str.match(r"^\d{5}"), "StockCode"].value_counts()

StockCode
POST         2122
DOT          1446
M            1421
C2            282
D             177
             ... 
DCGS0057        1
DCGS0073        1
DCGS0071        1
DCGS0066P       1
DCGS0067        1
Name: count, Length: 62, dtype: int64

In [11]:
country_count = df["Country"].value_counts()
country_pct = (df["Country"].value_counts(normalize=True) * 100).map(lambda x: f"{x:.2f}%")

pd.DataFrame({
    "Count": country_count,
    "Proportion": country_pct
})

,Count,Proportion
Country,,
United Kingdom,981330,91.94%
EIRE,17866,1.67%
Germany,17624,1.65%
France,14330,1.34%
Netherlands,5140,0.48%
Spain,3811,0.36%
Switzerland,3189,0.30%
Belgium,3123,0.29%
Portugal,2620,0.25%


In [12]:
print(df["Customer ID"].nunique(), df["Invoice"].nunique(), df["StockCode"].nunique())

5942 53628 5305


Temuan -> Keputusan
- Customer ID kosong 243007 atau 22,8% baris -> dibuang (tidak bisa dilacak), dicatat sebagai limitation
- Duplikat 34.335 baris -> drop_duplicates
- Invoice berawalan C = pembatalan -> dipisah ke tabel returns
- Quantity <= 0 dan Price <= 0 -> bukan penjualan valid, dibuang dari sales
- StockCode non-produk (POST, DOT, M, dst) -> hanya simpan kode 5 digit + huruf opsional
- Nilai ekstrem -> pakai model berbasis pohon